In [1]:
import networkx as nx
import random
import time
import csv
from itertools import chain
import pulp
import pandas as pd

In [2]:
def greedy_MIS(G):

    I = set()
    U = set(G.nodes())
    
    nodes = list(G.nodes())
    random.shuffle(nodes)
    
    for v in nodes:
        if v in U:
            I.add(v)
            U.remove(v)
            for u in G.neighbors(v):
                U.discard(u)
                
    return I 

In [3]:
def neighbor_cover(G, weights=None):
    
    if weights is None:
        weights = {v: 1 for v in G.nodes()}
    C, I = set(), set()
    U = set(G.nodes())
    
    while U:
        total_w = sum(weights[v] for v in U)
        pick = random.uniform(0, total_w)
        cumulative = 0
        for v in U:
            cumulative += weights[v]
            if cumulative >= pick:
                break
        I.add(v)
        U.remove(v)
        for u in G.neighbors(v):
            if u in U:
                C.add(u)
                U.remove(u)
                
    return C

In [4]:
def parallel_neighbor_cover(G):
    
    C, I = set(), set()
    U = set(G.nodes())
    perm = {v: random.random() for v in G.nodes()}
    
    while U:
        W = {u for u in U if all(perm[u] < perm[v]
                                for v in G.neighbors(u) if v in U)}
        I |= W
        U -= W
        neighs = set(chain.from_iterable(G.neighbors(u) for u in W)) & U
        C |= neighs
        U -= neighs
        
    return C

In [5]:
def lp_vertex_cover(G):
    
    V = list(G.nodes())
    E = list(G.edges())
    c = {v: 1 for v in V}

    prob = pulp.LpProblem("VC_LP", pulp.LpMinimize)
    x = {v: pulp.LpVariable(f"x_{v}", lowBound=0, upBound=1)
         for v in V}

    prob += pulp.lpSum(c[v] * x[v] for v in V)
    for u, v in E:
        prob += x[u] + x[v] >= 1

    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    x_val = {v: x[v].value() for v in V}

    VC = {v for v in V if x_val[v] >= 0.5}
    
    return VC

In [6]:
VERTEX_LIST = [100, 200, 500, 1000, 1500, 2000, 5000]
PROB_LIST = [0.03, 0.02, 0.01, 0.005, 0.003, 0.002, 0.0005]
TRIALS = 3

In [7]:
filename = "vertex_cover_results.csv"
with open(filename, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Algorithm", "Vertices", "Probability", "VC_Size", "Runtime"])

    for n in VERTEX_LIST:
        
        for p in PROB_LIST:
            
            for t in range(TRIALS):

                G = nx.erdos_renyi_graph(n, p)

                start = time.time()
                I = greedy_MIS(G)
                C = set(G.nodes()) - I
                runtime = time.time() - start
                writer.writerow(["GreedyMIS", n, p, len(C), runtime])


                start = time.time()
                C = neighbor_cover(G)
                runtime = time.time() - start
                writer.writerow(["NeighborCover", n, p, len(C), runtime])

             
                start = time.time()
                C = parallel_neighbor_cover(G)
                runtime = time.time() - start
                writer.writerow(["ParallelNeighborCover", n, p, len(C), runtime])

             
                start = time.time()
                C = lp_vertex_cover(G)
                runtime = time.time() - start
                writer.writerow(["LP_Rounding", n, p, len(C), runtime])

print("Results saved to:", filename)


Results saved to: vertex_cover_results.csv


In [8]:
import pandas as pd

df = pd.read_csv("vertex_cover_results.csv")

group_cols = ["Vertices", "Probability"]

winners_smallest = (
    df.loc[df.groupby(group_cols)["VC_Size"].idxmin()]
      .Algorithm.value_counts()
)

winners_fastest = (
    df.loc[df.groupby(group_cols)["Runtime"].idxmin()]
      .Algorithm.value_counts()
)

print("\n===============================")
print(" ALGORITHM PERFORMANCE SUMMARY")
print("===============================\n")

print("Algorithm that found the SMALLEST vertex cover MOST OFTEN:\n")
print(winners_smallest)

print("\nAlgorithm that was the FASTEST MOST OFTEN:\n")
print(winners_fastest)

summary = pd.DataFrame({
    "SmallestCoverWins": winners_smallest,
    "FastestWins": winners_fastest
}).fillna(0)

summary.to_csv("algorithm_performance_summary.csv")

print("\nSummary saved to algorithm_performance_summary.csv\n")
print(summary)



 ALGORITHM PERFORMANCE SUMMARY

Algorithm that found the SMALLEST vertex cover MOST OFTEN:

Algorithm
LP_Rounding              16
GreedyMIS                14
ParallelNeighborCover    13
NeighborCover             6
Name: count, dtype: int64

Algorithm that was the FASTEST MOST OFTEN:

Algorithm
GreedyMIS                46
ParallelNeighborCover     3
Name: count, dtype: int64

Summary saved to algorithm_performance_summary.csv

                       SmallestCoverWins  FastestWins
Algorithm                                            
GreedyMIS                             14         46.0
LP_Rounding                           16          0.0
NeighborCover                          6          0.0
ParallelNeighborCover                 13          3.0


In [1]:
import pandas as pd

df = pd.read_csv("vertex_cover_results.csv")

group_cols = ["Vertices", "Probability"]

winners_smallest = (
    df.loc[df.groupby(group_cols)["VC_Size"].idxmin()]
      .Algorithm.value_counts()
)

winners_fastest = (
    df.loc[df.groupby(group_cols)["Runtime"].idxmin()]
      .Algorithm.value_counts()
)

print("\n===============================")
print(" ALGORITHM PERFORMANCE SUMMARY")
print("===============================\n")

print("Algorithm that found the SMALLEST vertex cover MOST OFTEN:\n")
print(winners_smallest)

print("\nAlgorithm that was the FASTEST MOST OFTEN:\n")
print(winners_fastest)

summary = pd.DataFrame({
    "SmallestCoverWins": winners_smallest,
    "FastestWins": winners_fastest
}).fillna(0)

summary.to_csv("algorithm_performance_summary.csv")

print("\nSummary saved to algorithm_performance_summary.csv\n")
print(summary)



 ALGORITHM PERFORMANCE SUMMARY

Algorithm that found the SMALLEST vertex cover MOST OFTEN:

Algorithm
LP_Rounding              16
GreedyMIS                14
ParallelNeighborCover    13
NeighborCover             6
Name: count, dtype: int64

Algorithm that was the FASTEST MOST OFTEN:

Algorithm
GreedyMIS                46
ParallelNeighborCover     3
Name: count, dtype: int64

Summary saved to algorithm_performance_summary.csv

                       SmallestCoverWins  FastestWins
Algorithm                                            
GreedyMIS                             14         46.0
LP_Rounding                           16          0.0
NeighborCover                          6          0.0
ParallelNeighborCover                 13          3.0
